# BERDL Lakehouse — In-Cluster Ingest: {TENANT}_{DATASET}

Runs entirely within JupyterHub. No SSH tunnels or proxy required.

**Workflow:**
1. Fill in **Configuration** and run it.
2. Run **Setup** — connects Spark and MinIO.
3. Run **Schema** — review column types.
4. Run **Upload** — copies source files to MinIO bronze.
5. Run **Ingest** — writes Delta tables to silver.
6. Run **Verify** — confirms row counts match source files.
7. Run **Metadata** — second push marking tables completed.

### Configuration

Set these values before running any other cell.

In [ ]:
from pathlib import Path
import json as _json

# ── USER CONFIGURATION ───────────────────────────────────────────────
DATA_DIR        = Path("{DATA_DIR}")   # absolute path to source files on JH filesystem
TENANT          = "{TENANT}"           # Lakehouse tenant name (ignored when USER_TENANT=True)
DATASET         = "{DATASET}"          # dataset name (or None to use DATA_DIR.name)
BUCKET          = "cdm-lake"
MODE            = "{MODE}"             # "overwrite" or "append"
USER_TENANT     = False                # True = store in your personal user-tenant space
# ────────────────────────────────────────────────────────────────────

DATASET = DATASET or DATA_DIR.name

if USER_TENANT:
    import os
    _username  = os.environ.get("JUPYTERHUB_USER", "")
    if not _username:
        raise RuntimeError("JUPYTERHUB_USER env var not set — cannot resolve user-tenant namespace")
    NAMESPACE     = f"u_{_username}__{DATASET}"
    BRONZE_PREFIX = f"users-general-warehouse/{_username}/data/{DATASET}"
    SILVER_BASE   = f"s3a://{BUCKET}/users-sql-warehouse/{_username}/{NAMESPACE}.db"
else:
    _username     = None
    NAMESPACE     = f"{TENANT}_{DATASET}"
    BRONZE_PREFIX = f"tenant-general-warehouse/{TENANT}/datasets/{DATASET}"
    SILVER_BASE   = f"s3a://{BUCKET}/tenant-sql-warehouse/{TENANT}/{NAMESPACE}.db"

METADATA_PREFIX = f"{BRONZE_PREFIX}/metadata"
PROGRESS_KEY    = f"{BRONZE_PREFIX}/_ingest_progress.jsonl"

print(f"Mode         : {MODE}")
if USER_TENANT:
    print(f"Destination  : user-tenant space  →  namespace: {NAMESPACE}")
    print(f"               (private to your account)")
else:
    print(f"Destination  : shared tenant  →  namespace: {NAMESPACE}")
print(f"Bronze       : s3a://{BUCKET}/{BRONZE_PREFIX}/")
print(f"Silver       : {SILVER_BASE}")
print(f"Metadata     : s3a://{BUCKET}/{METADATA_PREFIX}/")

### Setup — Spark and MinIO

Connects directly to the cluster Spark session and MinIO. No tunnels required.

In [ ]:
import sys, csv, sqlite3, yaml
import pandas as pd
from pathlib import Path

# Bootstrap ingest_lib from the BERIL repo scripts/ directory
_found = False
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / "scripts" / "ingest_lib.py").exists():
        sys.path.insert(0, str(_p / "scripts"))
        _found = True
        break
if not _found:
    raise RuntimeError(
        "Could not find scripts/ingest_lib.py. "
        "Run this notebook from within the BERIL-research-observatory repo."
    )

from ingest_lib import get_spark_session, get_minio_client, parse_sql_schema, export_sqlite
from data_lakehouse_ingest import ingest

spark        = get_spark_session()
minio_client = get_minio_client()

print("Spark and MinIO ready.")

### Schema — Detect source files and parse column types

Scans `DATA_DIR` for TSV, CSV, Parquet, or SQLite files. If a `.sql` schema file is present
it is parsed and column types mapped to Spark SQL types. Without a schema file all columns
default to `STRING`.

In [ ]:
import os, glob

# Detect source format
db_files      = sorted(DATA_DIR.glob("*.db")) + sorted(DATA_DIR.glob("*.sqlite"))
tsv_files     = sorted(DATA_DIR.glob("*.tsv"))
csv_files     = sorted(DATA_DIR.glob("*.csv"))
parquet_files = sorted(DATA_DIR.glob("*.parquet"))
sql_files     = sorted(DATA_DIR.glob("*.sql"))

if parquet_files:
    SOURCE_MODE = "parquet"
    data_files  = parquet_files
    FILE_EXT    = ".parquet"
    DELIMITER   = None
elif db_files:
    SOURCE_MODE = "sqlite"
    SOURCE_DB   = db_files[0]
    data_files  = []  # populated after export
    FILE_EXT    = ".tsv"
    DELIMITER   = "\t"
elif tsv_files:
    SOURCE_MODE = "tsv"
    data_files  = tsv_files
    FILE_EXT    = ".tsv"
    DELIMITER   = "\t"
elif csv_files:
    SOURCE_MODE = "csv"
    data_files  = csv_files
    FILE_EXT    = ".csv"
    DELIMITER   = ","
else:
    raise FileNotFoundError(f"No supported source files found in {DATA_DIR}")

SQL_SCHEMA = sql_files[0] if sql_files else None

print(f"Source mode  : {SOURCE_MODE}")
print(f"Source files : {[f.name for f in data_files] if data_files else 'SQLite (export pending)'}")
print(f"Schema file  : {SQL_SCHEMA.name if SQL_SCHEMA else 'none — columns default to STRING'}")

In [ ]:
# Parse schema from .sql file (skip for Parquet — schema is embedded)
if SOURCE_MODE == "parquet":
    SCHEMAS     = {}
    SCHEMA_DEFS = {}
    print("Parquet mode — schema embedded in files.")
elif SQL_SCHEMA:
    SCHEMAS, SCHEMA_DEFS = parse_sql_schema(SQL_SCHEMA)
    print(f"Parsed schema for {len(SCHEMAS)} table(s): {list(SCHEMAS.keys())}")
else:
    SCHEMAS     = {}
    SCHEMA_DEFS = {}
    print("No schema file — all columns will default to STRING.")

# Export SQLite tables to TSV if needed
if SOURCE_MODE == "sqlite":
    data_files = export_sqlite(SOURCE_DB, DATASET, SCHEMAS)
    print(f"Exported {len(data_files)} table(s) from SQLite.")

### Upload — Copy source files to MinIO bronze

Uploads each source file to `s3a://cdm-lake/{BRONZE_PREFIX}/`. Files already present
at the correct size are skipped. Re-run safely if interrupted.

In [ ]:
import os

# Upload schema SQL if present
if SQL_SCHEMA:
    schema_key = f"{BRONZE_PREFIX}/config/{SQL_SCHEMA.name}"
    minio_client.fput_object(BUCKET, schema_key, str(SQL_SCHEMA))
    print(f"Uploaded schema: s3a://{BUCKET}/{schema_key}")

# Upload source data files
for f in data_files:
    key      = f"{BRONZE_PREFIX}/{f.name}"
    size_mb  = f.stat().st_size / 1_048_576
    # Skip if already present at the same size
    try:
        stat = minio_client.stat_object(BUCKET, key)
        if stat.size == f.stat().st_size:
            print(f"  SKIP  {f.name}  ({size_mb:.1f} MB — already in MinIO)")
            continue
    except Exception:
        pass
    print(f"  UPLOAD {f.name}  ({size_mb:.1f} MB) ...", end=" ", flush=True)
    minio_client.fput_object(BUCKET, key, str(f))
    print("done")

print(f"\nBronze path: s3a://{BUCKET}/{BRONZE_PREFIX}/")

### Ingest — Write Delta tables to silver

Calls `ingest()` for each table. Spark reads source files directly from MinIO bronze
and writes Delta tables to silver. Progress is logged to MinIO after each table so
re-running this cell resumes from where it left off.

In [ ]:
import json as _json
from datetime import datetime, timezone

PROGRESS_LOG = []

# Load existing progress log (enables resume on re-run)
try:
    resp = minio_client.get_object(BUCKET, PROGRESS_KEY)
    for line in resp.read().decode().splitlines():
        line = line.strip()
        if line:
            PROGRESS_LOG.append(_json.loads(line))
    completed_tables = {e["table"] for e in PROGRESS_LOG if e.get("status") == "complete"}
    print(f"Resuming — {len(completed_tables)} table(s) already complete: {completed_tables}")
except Exception:
    completed_tables = set()
    print("No existing progress log — starting fresh.")

for f in data_files:
    table_name = f.stem
    if table_name in completed_tables:
        print(f"  SKIP  {table_name} (already complete)")
        continue

    bronze_path = f"s3a://{BUCKET}/{BRONZE_PREFIX}/{f.name}"
    schema_sql  = SCHEMA_DEFS.get(table_name)

    cfg = {
        "dataset": DATASET,
        "paths": {
            "bronze_base": f"s3a://{BUCKET}/{BRONZE_PREFIX}/",
        },
        "tables": [
            {
                "name":       table_name,
                "bronze_path": bronze_path,
                "format":     FILE_EXT.lstrip("."),
                "mode":       MODE,
            }
        ],
    }
    if not USER_TENANT:
        cfg["tenant"] = TENANT
    if schema_sql:
        cfg["tables"][0]["schema_sql"] = schema_sql

    print(f"  INGEST {table_name} from {bronze_path} ...", end=" ", flush=True)
    report = ingest(cfg, spark=spark, minio_client=minio_client)
    if not report.get("success"):
        raise RuntimeError(f"Ingest failed for {table_name}: {report.get('errors')}")

    rows = report["tables"][0]["rows_written"]
    print(f"done — {rows:,} rows")

    # Append to progress log
    entry = {"table": table_name, "status": "complete",
             "rows_written": rows, "timestamp": datetime.now(timezone.utc).isoformat()}
    PROGRESS_LOG.append(entry)
    import io
    log_bytes = ("\n".join(_json.dumps(e) for e in PROGRESS_LOG) + "\n").encode()
    minio_client.put_object(BUCKET, PROGRESS_KEY, io.BytesIO(log_bytes), length=len(log_bytes))

print("\nIngest complete.")

### Verify — Row counts

In [ ]:
print(f"Verifying row counts in namespace: {NAMESPACE}\n")
all_ok = True
for f in data_files:
    table_name = f.stem
    # Expected: count lines minus header
    if FILE_EXT in (".tsv", ".csv"):
        with open(f) as fh:
            expected = sum(1 for _ in fh) - 1
    else:
        expected = None  # Parquet: trust Spark count

    actual = spark.sql(f"SELECT COUNT(*) AS n FROM {NAMESPACE}.{table_name}").collect()[0]["n"]
    if expected is None or actual == expected:
        print(f"  OK      {table_name}: {actual:,} rows")
    else:
        print(f"  MISMATCH {table_name}: expected {expected:,}, got {actual:,}")
        all_ok = False

print()
print("All tables OK." if all_ok else "WARNING: row count mismatches detected — check progress log.")
spark.stop()

### Metadata — Second push (completed)

Updates each `DATA_DIR/metadata/{table}.yaml` with `status: completed` and
`ingest_completed_at`, then uploads all files to MinIO.

In [ ]:
import json as _json, yaml

TABLE_COMPLETED_AT = {
    e["table"]: e["timestamp"]
    for e in PROGRESS_LOG
    if e.get("status") == "complete"
}

metadata_dir = DATA_DIR / "metadata"
n_updated = n_missing = 0

print("Updating metadata (second push — completed/failed)...")
for f in data_files:
    table_name = f.stem
    meta_path  = metadata_dir / f"{table_name}.yaml"
    if not meta_path.exists():
        print(f"  WARNING: {meta_path.name} not found locally — skipping {table_name}")
        n_missing += 1
        continue

    with open(meta_path) as fh:
        meta = yaml.safe_load(fh)

    completed_at          = TABLE_COMPLETED_AT.get(table_name)
    meta["ingest_completed_at"] = completed_at
    meta["status"]        = "completed" if completed_at else "failed"

    with open(meta_path, "w") as fh:
        yaml.dump(meta, fh, default_flow_style=False, allow_unicode=True, sort_keys=False)

    meta_key = f"{METADATA_PREFIX}/{table_name}.yaml"
    minio_client.fput_object(BUCKET, meta_key, str(meta_path))
    print(f"  {table_name}: status={meta['status']}  completed_at={completed_at}")
    n_updated += 1

print(f"\nMetadata second push complete — {n_updated} table(s) updated"
      + (f", {n_missing} missing local file(s)" if n_missing else ""))
print(f"  → s3a://{BUCKET}/{METADATA_PREFIX}/")
print(f"\nBronze: s3a://{BUCKET}/{BRONZE_PREFIX}/")
print(f"Silver: {SILVER_BASE}")